## CNN Checkpoint to S3 with an "everything" Manifest

A compact end-to-end example that builds a small CNN in PyTorch and stores the whole training state in AWS S3 via LAILA.

### What this notebook does
- Build a simple CNN model in `torch` and an optimizer over it
- Wrap them with `laila.addons.torch.Module` / `Optimizer` (same call API as the torch objects)
- Put model, optimizer and config under one `laila.complex_manifest` named `everything`
- Memorize `everything` with one call: every tensor becomes a detached CPU entry, the architecture a weight-free skeleton
- Remember `everything` back by nickname (its members arrive typed -- recursive blueprint resolution) and `rebuild()` the live objects
- Verify the round trip and clean up

### Credentials
This notebook reads AWS credentials from:
`laila.args`

In [ ]:
import laila
import torch
import torch.nn as nn

from laila.addons.torch import Module, Optimizer

In [ ]:
from laila.data import S3Pool

# These argument names are arbitrary and can be changed by the user.
# laila.args is just a container for user-provided arguments.
# Replace these placeholders with your own AWS credentials before running.
laila.args.AWS_BUCKET_NAME = "your-bucket-name"
laila.args.AWS_ACCESS_KEY_ID = "your-access-key-id"
laila.args.AWS_SECRET_ACCESS_KEY = "your-secret-access-key"
laila.args.AWS_REGION = "us-east-1"

s3_pool = S3Pool(
    bucket_name=laila.args.AWS_BUCKET_NAME,
    access_key_id=laila.args.AWS_ACCESS_KEY_ID,
    secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
    region_name=laila.args.AWS_REGION,
)


In [ ]:
laila.memory.extend(s3_pool, pool_nickname="cnn_weights_s3_pool")


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((8, 8)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)





In [ ]:
model = SimpleCNN()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# I DO SOME TRAINING HERE
for _ in range(3):
    optimizer.zero_grad()
    model(torch.randn(4, 3, 32, 32)).sum().backward()
    optimizer.step()

In [ ]:
final_model = Module(model.eval(), nickname="my_cnn")
final_optimizer = Optimizer(optimizer, nickname="my_cnn_optimizer")
print(final_model)

In [ ]:
everything = laila.complex_manifest(nickname="my_cnn_everything")
everything["model"] = final_model          # ModuleManifest: state_dict tensors + architecture skeleton
everything["optimizer"] = final_optimizer  # OptimizerManifest: state tensors + model / parameter references
everything["config"] = laila.constant(data={"lr": 0.01, "momentum": 0.9, "epochs": 3}, nickname="my_cnn_config")

for key in everything.keys():
    print(key, "->", everything.blueprint[key])

In [ ]:
with laila.guarantee:
    laila.memorize(everything, pool_nickname="cnn_weights_s3_pool")

print("Uploaded:", everything.global_id)

In [ ]:
reference = {name: tensor.clone() for name, tensor in final_model.state_dict().items()}
del model, optimizer, final_model, final_optimizer, everything

with laila.guarantee:
    ref = laila.remember("MANIFEST:COMPLEX:my_cnn_everything", pool_nickname="cnn_weights_s3_pool")
everything = ref.result

print({key: type(value).__name__ for key, value in everything.members.items()})

In [ ]:
live = everything.rebuild()
model, optimizer, config = live["model"], live["optimizer"], live["config"].data

assert all(torch.equal(reference[name], tensor) for name, tensor in model.state_dict().items())
print("Recovered model matches:", type(model).__name__, "over", type(model.module).__name__)
print("Optimizer:", type(optimizer.optimizer).__name__, "lr =", optimizer.param_groups[0]["lr"], "config:", config)

In [ ]:
with laila.guarantee:
    laila.forget(everything, pool_nickname="cnn_weights_s3_pool")

print("Cleanup complete.")